In [ ]:
!pip install -q transformers datasets evaluate sacrebleu
!pip install rouge_score bert-score
from google.colab import files
import pandas as pd

print(" Please upload your CSV dataset (columns required: original_abstract, human1, llm_question, better_question)")
uploaded = files.upload()

csv_path = list(uploaded.keys())[0]
df = pd.read_csv(csv_path)
print(f"Loaded dataset: {csv_path} ({len(df)} rows)")
import os, torch, numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments
)
import evaluate, sacrebleu

MODEL_NAME = "google/pegasus-large"
OUTPUT_DIR = "/content/pegasus_question_gen"

MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 64
EPOCHS = 4
BATCH_SIZE = 4
LR = 3e-5
USE_FP16 = True
NUM_BEAMS = 5
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

required_cols = ["original_abstract", "human1", "llm_question", "better_question"]
assert all(c in df.columns for c in required_cols), f"CSV must contain columns: {required_cols}"

df = df[required_cols].dropna().reset_index(drop=True)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED, shuffle=True)
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

SEP = " || "

def preprocess_map(batch):
    inputs, targets = [], []
    for oa, h1, lq, bq in zip(batch["original_abstract"], batch["human1"], batch["llm_question"], batch["better_question"]):
        inp = SEP.join([
            "Abstract: " + str(oa).strip(),
            "Human1: " + str(h1).strip(),
            "LLM_Q: " + str(lq).strip()
        ])
        inputs.append(inp)
        targets.append(str(bq))
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=MAX_TARGET_LEN, truncation=True, padding="max_length")
    model_inputs["labels"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in example]
        for example in labels["input_ids"]
    ]
    return model_inputs

train_ds = train_ds.map(preprocess_map, batched=True, remove_columns=train_ds.column_names)
test_ds = test_ds.map(preprocess_map, batched=True, remove_columns=test_ds.column_names)
train_ds.set_format(type="torch")
test_ds.set_format(type="torch")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    save_total_limit=2,
    logging_steps=100,
    eval_strategy="no",
    fp16=USE_FP16,
    push_to_hub=False,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer
)

print("Starting training...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training complete. Model saved to", OUTPUT_DIR)
rouge = evaluate.load("rouge")

def generate_text(oa, h1, lq, num_beams=NUM_BEAMS):
    inp = SEP.join([
        "Abstract: " + str(oa).strip(),
        "Human1: " + str(h1).strip(),
        "LLM_Q: " + str(lq).strip()
    ])
    enc = tokenizer(inp, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN, padding="longest").to(device)
    out = model.generate(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        max_new_tokens=48,
        num_beams=num_beams,
        early_stopping=True,
        no_repeat_ngram_size=2,
        length_penalty=1.0,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

preds, refs = [], []
for i in range(len(test_df)):
    oa = test_df.iloc[i]["original_abstract"]
    h1 = test_df.iloc[i]["human1"]
    lq = test_df.iloc[i]["llm_question"]
    tgt = test_df.iloc[i]["better_question"]
    pred = generate_text(oa, h1, lq)
    preds.append(pred)
    refs.append(tgt)
    if i < 5:
        print(f"\n  Input Abstract: {oa[:100].replace(chr(10),' ')}...")
        print("Human1:", h1)
        print("LLM_Q:", lq)
        print("Ground Truth:", tgt)
        print("Prediction:", pred)
        print("------")

rouge_res = rouge.compute(predictions=preds, references=refs)
print("\n ROUGE:", rouge_res)

print("SacreBLEU:", sacrebleu.corpus_bleu(preds, [refs]).score)

bertscore = evaluate.load("bertscore")
bert_res = bertscore.compute(predictions=preds, references=refs, lang="en")
print("BERTScore mean F1:", np.mean(bert_res["f1"]))

print("Done.")

 Please upload your CSV dataset (columns required: original_abstract, human1, llm_question, better_question)


Saving dataset - Sheet1 (3).csv to dataset - Sheet1 (3) (2).csv
Loaded dataset: dataset - Sheet1 (3) (2).csv (726 rows)
Using device: cuda


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/580 [00:00<?, ? examples/s]

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

...

Step,Training Loss
100,4.215500
200,3.338200
300,3.008100
400,2.872900
500,2.821900


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 256, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Training complete. Model saved to /content/pegasus_question_gen

  Input Abstract: we have developed a novel , publicly available annotation tool for the semantic encoding of texts , ...
Human1: We have created a new annotation (publicly available) tool intended to be used in semantic encoding of texts and especially when it comes to the domain of narratives. The tool can be used to formalize text scopes as well as temporal connections and other narrative contents. It contains an appropriate natural-language generation module, which is capable of re-generating the text out of these formal structures, and the annotation procedure becomes easier, too. The Collection Experiment has demonstrated the fact that even a layman can successfully come up with semantic encodings of short fables using the tool. As a tool on its own, we offer not only the capacity to be rebuilt at ease as an investigative tool in the domain of semantics, especially in cases where they demand formal coding of text, e

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore mean F1: 0.8845871642027816
Done.


In [ ]:
import shutil
shutil.make_archive("/content/pegasus_question_gen", "zip", "/content/pegasus_question_gen")
